In [6]:
# Cliente local para enviar o prompt do pipeline aquario ao endpoint compatível com OpenAI.
# Este notebook não executa a requisição automaticamente; rode as células 4 e 5 quando o servidor estiver ativo.


# Solicitação local para o pipeline Qwen do projeto aquario

Este notebook monta o prompt completo e envia-o ao endpoint compatível com OpenAI em `http://localhost:20128/v1/chat/completions`.

O servidor local precisa estar ativo antes de executar a célula de requisição.

In [7]:
from pathlib import Path

PROJECT_ROOT = Path('/Users/fernandodavilalbcfilho/Downloads/aquario')
MODEL = 'gemini/gemini-3.6-flash'
ENDPOINT = 'http://localhost:20128/v1/chat/completions'

PROMPT = f'''Você é um engenheiro de software especialista em pipelines de imagem generativa e pixel art para jogos.

Trabalhe neste projeto local: {PROJECT_ROOT}

Objetivo: implementar ou continuar o pipeline que transforma fotos reais de espécies de aquário em sprites pixel art consistentes, usando exatamente Qwen/Qwen-Image com Diffusers e QwenImageImg2ImgPipeline.

Arquivos principais:
- JSON: {PROJECT_ROOT / 'kauar_peixes.json'}
- CSV: {PROJECT_ROOT / 'kauar_peixes.csv'}
- runner: {PROJECT_ROOT / 'aquarium/qwen_pixelart.py'}
- CLI: {PROJECT_ROOT / 'scripts/qwen_pixelart.py'}
- config Qwen: {PROJECT_ROOT / 'config/qwen_pixelart.json'}
- estilo: {PROJECT_ROOT / 'config/style.json'}
- smoke: {PROJECT_ROOT / 'config/smoke.json'}
- guia: {PROJECT_ROOT / 'docs/QWEN_PIXELART.md'}
- notebook GPU/Colab: {PROJECT_ROOT / 'notebooks/Qwen_Pixel_Art.ipynb'}
- fotos/cache: {PROJECT_ROOT / 'data/sources'}
- saída: {PROJECT_ROOT / 'output/teste-qwen'}

Use exatamente o modelo Qwen/Qwen-Image, diffusers.QwenImageImg2ImgPipeline e inferência img2img com a foto preparada no argumento image.

Comandos esperados:
1. Dry-run sem GPU e sem pesos:
python3 {PROJECT_ROOT / 'scripts/qwen_pixelart.py'} --dataset {PROJECT_ROOT / 'kauar_peixes.json'} --smoke --dry-run
2. Smoke CUDA:
python3 {PROJECT_ROOT / 'scripts/qwen_pixelart.py'} --dataset {PROJECT_ROOT / 'kauar_peixes.json'} --smoke --remove-background --output {PROJECT_ROOT / 'output/teste-qwen'}
3. Foto avulsa:
python3 {PROJECT_ROOT / 'scripts/qwen_pixelart.py'} --input {PROJECT_ROOT / 'data/sources/fd6fb795dbc0c857.png'} --subject "Brachygobius doriae, small yellow aquarium fish with black vertical stripes" --strength 0.65 --remove-background --output {PROJECT_ROOT / 'output/meu-peixe'}
4. Catálogo, somente após validar smoke:
python3 {PROJECT_ROOT / 'scripts/qwen_pixelart.py'} --dataset {PROJECT_ROOT / 'kauar_peixes.json'} --all --remove-background --output {PROJECT_ROOT / 'output/catalogo-qwen'}

Prompt base: Create a clean 16-bit pixel art aquarium sprite of {{subject}}, preserving the real animal or plant silhouette, main colors, distinctive markings, fins, antennae, leaves or roots. Single isolated subject, orthographic side view for fish and shrimp, upright natural pose for aquarium plants, transparent-looking white studio background, crisp pixel clusters, limited palette, dark navy 1 pixel outline, upper-left soft aquarium light, no scenery, no substrate, no bubbles, no text, centered with comfortable margins.

Negative prompt: photorealistic, realistic photo, blurry, smooth gradients, 3d render, watercolor, oil painting, text, watermark, logo, multiple subjects, cropped body, extra fins, deformed anatomy, background scenery, aquarium tank, gravel, bubbles, hands, price tag, checkerboard background

Critérios de aceite:
- dry-run funciona neste Mac sem GPU;
- CUDA smoke gera pelo menos 3 espécies;
- cada item salva .input.png, .raw.png, .png final 96x96 RGBA, .preview.png e .json;
- sprite final tem fundo transparente, contorno consistente, 96x96 e sem cenário;
- falhas por item são registradas sem interromper o lote.

Antes de editar, inspecione o código atual, formule uma hipótese local verificável, faça alterações mínimas, rode testes focados e reporte limitações de memória/dispositivo.'''

print(f'Prompt preparado: {len(PROMPT)} caracteres')


Prompt preparado: 3944 caracteres


In [8]:
import json

payload = {
    'model': MODEL,
    'stream': False,
    'messages': [{'role': 'user', 'content': PROMPT}],
}
payload_json = json.dumps(payload, ensure_ascii=False).encode('utf-8')

print(json.dumps({
    'endpoint': ENDPOINT,
    'model': MODEL,
    'stream': False,
    'prompt_characters': len(PROMPT),
}, ensure_ascii=False, indent=2))


{
  "endpoint": "http://localhost:20128/v1/chat/completions",
  "model": "gemini/gemini-3.6-flash",
  "stream": false,
  "prompt_characters": 3944
}


In [9]:
from urllib.request import Request, urlopen

request = Request(
    ENDPOINT,
    data=payload_json,
    headers={
        'Content-Type': 'application/json',
        'Authorization': 'Bearer dummy-key',
    },
    method='POST',
)

with urlopen(request, timeout=1800) as response:
    response_body = response.read().decode('utf-8')

print(response_body)


{"id":"chatcmpl-i1qratWWFfiomtkPiZutwAE","object":"chat.completion","created":1789614749,"model":"gemini-3.6-flash","choices":[{"index":0,"finish_reason":"malformed_function_call","message":{"role":"assistant","content":"","reasoning_content":"**Initial Pipeline Inspection**\n\nOkay, so the user wants me to dive into a pipeline that's already underway – converting aquarium photos into pixel art sprites. The tools of choice are `Qwen/Qwen-Image`, `Diffusers`, and `QwenImageImg2ImgPipeline`, all running on some photos. Right, let's get started.\n\nFirst thing's first, I need to understand what's already in place. The project is at `/Users/fernandodavilalbcfilho/Downloads/aquario`, so let's start by exploring the directory structure. I need to get a lay of the land, list the files, and see how the existing implementation is structured. Shell commands are my friend here; I can use those to quickly understand the setup, look for configuration files, and identify the key scripts or notebooks

In [10]:
result = json.loads(response_body)

if result.get('error'):
    raise RuntimeError(json.dumps(result['error'], ensure_ascii=False, indent=2))

choices = result.get('choices', [])
if not choices:
    raise RuntimeError(f'Resposta sem choices. Chaves recebidas: {list(result)}')

choice = choices[0]
message = choice.get('message', {})
content = message.get('content') or ''
reasoning = message.get('reasoning_content') or ''
finish_reason = choice.get('finish_reason')

print(f'Finish reason: {finish_reason}')

if content.strip():
    print(content)
elif reasoning.strip():
    print('O servidor não colocou a resposta em content; exibindo reasoning_content:')
    print(reasoning)
    if finish_reason == 'malformed_function_call':
        print('\nAviso: o modelo tentou uma chamada de função malformada. Para obter uma resposta final limpa, desative tool/function calling no servidor ou use um modelo/configuração sem ferramentas.')
elif response_body.strip():
    print('Não foi encontrado content nem reasoning_content. Resposta completa:')
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print('O servidor retornou uma resposta vazia.')


Finish reason: malformed_function_call
O servidor não colocou a resposta em content; exibindo reasoning_content:
**Initial Pipeline Inspection**

Okay, so the user wants me to dive into a pipeline that's already underway – converting aquarium photos into pixel art sprites. The tools of choice are `Qwen/Qwen-Image`, `Diffusers`, and `QwenImageImg2ImgPipeline`, all running on some photos. Right, let's get started.

First thing's first, I need to understand what's already in place. The project is at `/Users/fernandodavilalbcfilho/Downloads/aquario`, so let's start by exploring the directory structure. I need to get a lay of the land, list the files, and see how the existing implementation is structured. Shell commands are my friend here; I can use those to quickly understand the setup, look for configuration files, and identify the key scripts or notebooks. I need to figure out what's been done, what needs attention, and where I can make improvements or modifications to fit the user's ult